In [2]:
!pip install transformers datasets torch evaluate rouge_score
!pip install -U datasets


  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-3.6.0-py3-none-any.whl (491 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [1]:
import wandb
wandb.init(mode="disabled")
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, pipeline
from datasets import load_dataset
import torch
import math

# 1. Load a text dataset (e.g., `imdb` for conversational-like data)
dataset = load_dataset("imdb")
small_train = dataset["train"].select(range(1000))
small_validation = dataset["test"].select(range(200))

# 2. Load the tokenizer and model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Ensure the tokenizer can handle padding
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    inputs = examples["text"]

    # Tokenize inputs
    tokenized = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    tokenized["labels"] = tokenized["input_ids"].clone()

    return tokenized

# Process datasets
tokenized_train = small_train.map(
    preprocess_function,
    batched=True,
    remove_columns=small_train.column_names,
    batch_size=8
)

tokenized_validation = small_validation.map(
    preprocess_function,
    batched=True,
    remove_columns=small_validation.column_names,
    batch_size=8
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation
)

# Train the model
trainer.train()

# Save the model
model.save_pretrained("./text_gen_model")
tokenizer.save_pretrained("./text_gen_model")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,4.442200
50,3.726100
100,3.617800
150,3.509000
200,3.406600
250,3.368600


('./text_gen_model/tokenizer_config.json',
 './text_gen_model/special_tokens_map.json',
 './text_gen_model/vocab.json',
 './text_gen_model/merges.txt',
 './text_gen_model/added_tokens.json',
 './text_gen_model/tokenizer.json')

In [2]:
# Test the model with a text generation pipeline
text_gen_pipeline = pipeline("text-generation", model="./text_gen_model", tokenizer="./text_gen_model")

# Example: Chatbot-like behavior
context = "User: How are you?\nBot: I am"
prediction = text_gen_pipeline(context, max_length=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

print(f"\nInput: {context}")
print(f"Response: {prediction[0]['generated_text']}")

# 1. Perplexity Evaluation
print("\n--- Perplexity Evaluation ---")
eval_results = trainer.evaluate()
print(f"Evaluation Loss: {eval_results['eval_loss']}")
print(f"Perplexity: {math.exp(eval_results['eval_loss'])}")

# 2. Qualitative Evaluation
test_contexts = [
    "User: Tell me about AI.\nBot:",
    "User: What's the weather like?\nBot:",
    "User: Recommend me a movie.\nBot:"
]

print("\n--- Qualitative Evaluation ---")
for context in test_contexts:
    prediction = text_gen_pipeline(context, max_length=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)
    print(f"\nInput: {context}")
    print(f"Response: {prediction[0]['generated_text']}")
import evaluate

# 3. BLEU/ROUGE Scores
print("\n--- BLEU/ROUGE Evaluation ---")
import evaluate
metric = evaluate.load("rouge")
# Generate predictions
predictions = [
    text_gen_pipeline(context, max_length=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
    for context in test_contexts
]

# Reference responses
references = [
    "Bot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.",
    "Bot: I don't know the weather right now, but you can check your local forecast!",
    "Bot: I recommend you watch 'Inception,' a thrilling movie with a unique plot."
]

results = metric.compute(predictions=predictions, references=references, use_stemmer=True)
print(results)




Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.



Input: User: How are you?
Bot: I am
Response: User: How are you?
Bot: I am not a bot, it is me. I am acting like a human being without emotion. I can understand your emotions.<br /><br />I can also say that you are my first "

--- Perplexity Evaluation ---


Evaluation Loss: 3.5407638549804688
Perplexity: 34.49325697658652

--- Qualitative Evaluation ---

Input: User: Tell me about AI.
Bot:
Response: User: Tell me about AI.
Bot: Don't bother telling me what it's about. There are already a lot of good movies out there about this one. The main problem is that it's a joke by definition. The movie is about

Input: User: What's the weather like?
Bot:
Response: User: What's the weather like?
Bot: I watched a few episodes of the "Stranger Things" so I was just about to mention it to someone else and the person who was supposed to be watching it wasn't going to be thinking

Input: User: Recommend me a movie.
Bot:
Response: User: Recommend me a movie.
Bot: This movie is so bad you could do anything else.

--- BLEU/ROUGE Evaluation ---


{'rouge1': np.float64(0.1721661054994388), 'rouge2': np.float64(0.012578616352201257), 'rougeL': np.float64(0.12300785634118967), 'rougeLsum': np.float64(0.1721661054994388)}


In [11]:
context = "I dont like you"
prediction = text_gen_pipeline(context, max_length=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)
print(f"\nInput: {context}")
print(f"Response: {prediction[0]['generated_text']}")


Input: I dont like you
Response: I dont like you guys so I hope you guys stop writing this kind of drama.<br /><br />Hey everyone who's seen these movies, and there's so many scenes and characters that you can't tell them apart from the other two.
